# Weight-only quantization for the CPU demo (onnx-weightonly)

**Settings:** Accelerator **None (CPU)**, Internet **ON**.
**Inputs:** Your Work -> Notebooks -> **onnx-export** (the exported fp32 ONNX) **and** **llm-cls-train-v2** (tokenizer + val rows).

**Why:** in `onnx-export`, plain dynamic INT8 quantized the *activations* too, and LLM activation outliers hurt accuracy
(onnx int8 +0.011 log loss, torch int8 nearly broken). Weight-only quantization compresses only the **weights**
(4-bit or 8-bit, per block of 32 values) and keeps activations in fp32. That's the approach behind AWQ / GPTQ / llama.cpp.

| Variant | What |
|---|---|
| onnx_int8_perchannel | Dynamic INT8 with a separate scale per output channel (better than per-tensor) |
| onnx_w8_block32 | Weight-only 8-bit, block size 32 |
| onnx_w4_block32 | Weight-only 4-bit, block size 32 |

Output: `cpu_engines_all.csv` (all engines incl. the earlier ones) and `hf_model/` with the chosen engine.

In [ ]:
# keep huggingface_hub < 2.0: transformers 5.16 refuses to import with 2.x
# onnx-ir: needed by onnxruntime's weight-only (MatMulNBits) quantizer
!pip install -q -U onnx onnx-ir onnxruntime "huggingface_hub>=1.5,<2" 2>&1 | tail -n 2
!pip list 2>/dev/null | grep -E "^(transformers|huggingface-hub|onnx|onnx-ir|onnxruntime) "

In [ ]:
import os, gc, glob, json, time, shutil, traceback
import numpy as np, pandas as pd
import onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from sklearn.metrics import log_loss
from transformers import AutoTokenizer

def find(p):
    hits = glob.glob(f"/kaggle/input/**/{p}", recursive=True)
    assert hits, f"input not found: {p} (added both inputs?)"
    return hits[0]

FP32_IN = find("model_fp32.onnx")
os.makedirs("/tmp/fp32", exist_ok=True)                  # /kaggle/input is read-only; quantizers write temp files next to the model
for f in glob.glob(FP32_IN + "*"):                        # .onnx + its external weights file(s)
    shutil.copy(f, "/tmp/fp32/")
FP32 = "/tmp/fp32/" + os.path.basename(FP32_IN)
MODEL_DIR = os.path.dirname(find("qwen-cls/config.json"))
INFO = json.load(open(find("train_info.json")))
PREV = pd.read_csv(find("cpu_engines.csv"))
OUT = "/kaggle/working"; QDIR = f"{OUT}/quant"; os.makedirs(QDIR, exist_ok=True)
THREADS = os.cpu_count(); N_EVAL = 80
print("fp32 onnx:", FP32, "| threads:", THREADS)
display(PREV)

tok = AutoTokenizer.from_pretrained(MODEL_DIR)
HEADERS = dict(prompt="Prompt: ", a="\n\nResponse A: ", b="\n\nResponse B: ", q="\n\nWhich response is better?")
H = {k: tok(v, add_special_tokens=False)["input_ids"] for k, v in HEADERS.items()}
def encode(p, a, b):
    T = lambda s, n: tok(s, add_special_tokens=False, truncation=True, max_length=n)["input_ids"]
    return H["prompt"] + T(p, INFO["max_p"]) + H["a"] + T(a, INFO["max_r"]) + H["b"] + T(b, INFO["max_r"]) + H["q"]
val = pd.read_parquet(find("val.parquet")).iloc[:N_EVAL]          # same 80 rows as onnx-export
ROWS = [encode(p, a, b) for p, a, b in zip(val.prompt, val.response_a, val.response_b)]
Y = val["label"].values

def ort_runner(path, threads=THREADS):
    so = ort.SessionOptions(); so.intra_op_num_threads = threads
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
    def one(ids):
        x = np.array([ids], dtype=np.int64)
        return sess.run(["probs"], {"input_ids": x, "attention_mask": np.ones_like(x)})[0][0]
    return sess, one

def size_gb(path):
    return round(sum(os.path.getsize(f) for f in glob.glob(path + "*")) / 1e9, 2)

RESULTS = []
def evaluate(name, path, **extra):
    sess, one = ort_runner(path)
    one(ROWS[0])
    probs, ms = [], []
    for ids in ROWS:
        t = time.perf_counter(); probs.append(one(ids)); ms.append((time.perf_counter() - t) * 1000)
    probs = np.array(probs, dtype=np.float32)
    del sess; gc.collect()
    _, one2 = ort_runner(path, threads=2)                          # free Space has 2 vCPUs
    ms2 = []
    for ids in ROWS[:10]:
        t = time.perf_counter(); one2(ids); ms2.append((time.perf_counter() - t) * 1000)
    r = dict(engine=name, rows=len(ROWS), p50_ms=round(float(np.median(ms))), p95_ms=round(float(np.percentile(ms, 95))),
             p50_ms_2threads=round(float(np.median(ms2))), size_gb=size_gb(path),
             log_loss=round(log_loss(Y, probs, labels=[0, 1, 2]), 4), acc=round(float((probs.argmax(1) == Y).mean()), 4), **extra)
    RESULTS.append(r); print(r)
    gc.collect()

## 1. Per-channel dynamic INT8

In [ ]:
P_INT8PC = f"{QDIR}/model_int8_pc.onnx"
try:
    t = time.time()
    quantize_dynamic(FP32, P_INT8PC, weight_type=QuantType.QInt8, per_channel=True, use_external_data_format=True)
    evaluate("onnx_int8_perchannel", P_INT8PC, quantize_s=round(time.time() - t))
except Exception as e:
    traceback.print_exc(); RESULTS.append(dict(engine="onnx_int8_perchannel", error=repr(e)[:300]))

## 2. Weight-only 8-bit and 4-bit (MatMulNBits)

In [ ]:
def nbits_quantize(src, dst, bits, block_size=32):
    model = onnx.load(src)                                         # loads the external weights too
    try:
        from onnxruntime.quantization.matmul_nbits_quantizer import MatMulNBitsQuantizer
        try:
            q = MatMulNBitsQuantizer(model, bits=bits, block_size=block_size, is_symmetric=True)
        except TypeError:                                          # older signature without `bits`
            assert bits == 4, "this onnxruntime only supports 4-bit weight-only"
            q = MatMulNBitsQuantizer(model, block_size=block_size, is_symmetric=True)
    except ImportError:
        from onnxruntime.quantization.matmul_4bits_quantizer import MatMul4BitsQuantizer
        assert bits == 4, "this onnxruntime only supports 4-bit weight-only"
        q = MatMul4BitsQuantizer(model, block_size=block_size, is_symmetric=True)
    q.process()
    n_q = sum(1 for nd in q.model.model.graph.node if nd.op_type == "MatMulNBits")
    q.model.save_model_to_file(dst, use_external_data_format=True)
    del model, q; gc.collect()
    return n_q

for bits in (8, 4):
    name = f"onnx_w{bits}_block32"
    path = f"{QDIR}/model_w{bits}.onnx"
    try:
        t = time.time()
        n_q = nbits_quantize(FP32, path, bits)
        print(f"{name}: {n_q} MatMul layers quantized in {time.time() - t:.0f}s")
        assert n_q > 0, "no MatMul was quantized (weights not stored as constants?)"
        evaluate(name, path, quantize_s=round(time.time() - t), quantized_matmuls=n_q)
    except Exception as e:
        traceback.print_exc(); RESULTS.append(dict(engine=name, error=repr(e)[:300]))

## 3. All CPU engines + pick the demo engine
Rule (same as before): fastest **2-thread** engine whose log loss stays within **+0.01** of fp32.

In [ ]:
new = pd.DataFrame(RESULTS)
allres = pd.concat([PREV, new], ignore_index=True)
display(allres); allres.to_csv(f"{OUT}/cpu_engines_all.csv", index=False)

ref = float(PREV.loc[PREV.engine == "torch_fp32", "log_loss"].iloc[0])
cand = new[new["error"].isna()] if "error" in new else new
if "log_loss" in cand:
    cand = cand.dropna(subset=["log_loss"])
    cand = cand[cand.log_loss <= ref + 0.01].sort_values("p50_ms_2threads")
else:
    cand = cand.iloc[0:0]                                 # every variant failed -> keep onnx_fp32
CHOSEN = cand.engine.iloc[0] if len(cand) else "onnx_fp32"
print("CHOSEN demo engine:", CHOSEN)

src = {"onnx_fp32": FP32, "onnx_int8_perchannel": P_INT8PC,
       "onnx_w8_block32": f"{QDIR}/model_w8.onnx", "onnx_w4_block32": f"{QDIR}/model_w4.onnx"}[CHOSEN]
HF = f"{OUT}/hf_model"; shutil.rmtree(HF, ignore_errors=True); os.makedirs(HF)
for f in glob.glob(src + "*"):
    shutil.copy(f, HF)
tok.save_pretrained(HF)
row = allres[allres.engine == CHOSEN].iloc[0].to_dict() if CHOSEN in set(allres.engine) else {}
cfg = dict(engine="onnxruntime", source_engine=CHOSEN, onnx_file=os.path.basename(src),
           max_p=INFO["max_p"], max_r=INFO["max_r"], headers=HEADERS, labels=["Response A wins", "Response B wins", "Tie"],
           base_model=INFO.get("model_id"), train_rows=INFO.get("train_rows"), val_log_loss_gpu=INFO.get("val_log_loss"),
           expected_space_p50_ms_2threads=row.get("p50_ms_2threads"), cpu_results=allres.fillna("").to_dict("records"))
json.dump(cfg, open(f"{HF}/demo_config.json", "w"), indent=1, default=str)
!du -sh {HF}; ls -la {HF}

## 4. (Optional) push `hf_model/` to the Hugging Face Hub
Add a **write** token as Kaggle secret `HF_TOKEN` (Add-ons -> Secrets), set `PUSH = True` and your repo name.

In [ ]:
PUSH = False
HF_REPO = "YOUR_HF_USERNAME/chatbot-arena-preference-judge"     # <- change
if PUSH:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi
    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    api.create_repo(HF_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=f"{OUT}/hf_model", repo_id=HF_REPO, repo_type="model")
    print("uploaded -> https://huggingface.co/" + HF_REPO)